# Day 05 — Tổng hợp pipeline: tạo bảng và hình bằng code

**Slides (tải về):** [`day05_slides.pptx`](_static/slides/day05_slides.pptx)

Mục tiêu:
- Ghép toàn bộ các bước phân tích thành một notebook chạy lại được.
- Tạo bảng và hình theo đúng flow báo cáo.
- Xuất thư mục outputs rõ ràng để kiểm tra và nộp.

Sản phẩm cuối buổi:
- outputs/day05/table1_overall.csv và table1_by_egfr.csv
- outputs/day05/roc_compare_rois.png
- outputs/day05/cv_results.csv và bootstrap_ci.csv
- outputs/day05/ngs_deltaP_table.csv và deltaP_bar.png
- outputs/day05/results_summary.md
- outputs/day05_submission.zip


## Setup
Chạy các cell theo thứ tự.

In [ ]:
# Nếu chạy trên Colab và thiếu thư viện, mở comment các dòng dưới đây.
# !pip -q install scikit-learn scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

from scipy.stats import mannwhitneyu

plt.rcParams["figure.figsize"] = (6, 4)

SEED = 42
np.random.seed(SEED)


## Bước 1 — Load dữ liệu demo

In [ ]:
# Notebook tự tìm file trong repo. Nếu chạy trên Colab và không thấy file, sẽ thử tải từ GitHub.
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"

def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out_path = Path(rel_path).name
    print("Tải file:", url)
    urllib.request.urlretrieve(url, out_path)
    return Path(out_path)

def load_csv(rel_path: str) -> pd.DataFrame:
    candidates = [
        Path(rel_path),
        Path("../")/rel_path,
        Path("../../")/rel_path,
        Path("data")/Path(rel_path).name,
        Path("../data")/Path(rel_path).name,
        Path("_static/data")/Path(rel_path).name,
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    # Thử tải từ GitHub
    p = download_from_github(rel_path)
    return pd.read_csv(p)

cohort = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
ngs64 = load_csv("data/ngs_pathway_demo_64.csv")

print("Cohort shape:", cohort.shape)
print("NGS subset shape:", ngs64.shape)

cohort.head()


### Kiểm tra nhanh cohort

In [ ]:
# Cột nhãn (label)
LABEL_COL = "egfr_mutation"
ID_COL = "patient_id"

assert LABEL_COL in cohort.columns
assert ID_COL in cohort.columns

print("Tỉ lệ EGFR dương tính:", cohort[LABEL_COL].mean().round(3))
print("Số bệnh nhân:", cohort.shape[0])


## Chuẩn bị thư mục outputs

In [ ]:
OUT_DIR = Path("outputs/day05")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR


## Bước 2 — Table 1 mô tả cohort

In [ ]:
# Một Table 1 đơn giản: tuổi, giới, EGFR
# (Có thể mở rộng thêm biến nếu dataset demo có)

def table1_overall(df: pd.DataFrame) -> pd.DataFrame:
    out = {}
    if "age" in df.columns:
        out["age_mean"] = [df["age"].mean()]
        out["age_std"] = [df["age"].std(ddof=1)]
    if "sex" in df.columns:
        out["male_percent"] = [(df["sex"] == "M").mean() * 100]
    out["egfr_positive_percent"] = [df[LABEL_COL].mean() * 100]
    return pd.DataFrame(out)

def table1_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for g, sub in df.groupby(group_col):
        row = {"group": int(g), "n": sub.shape[0]}
        if "age" in sub.columns:
            row["age_mean"] = sub["age"].mean()
            row["age_std"] = sub["age"].std(ddof=1)
        if "sex" in sub.columns:
            row["male_percent"] = (sub["sex"] == "M").mean() * 100
        rows.append(row)
    return pd.DataFrame(rows).sort_values("group")

t1_all = table1_overall(cohort)
t1_grp = table1_by_group(cohort, LABEL_COL)

t1_all.to_csv(OUT_DIR/"table1_overall.csv", index=False)
t1_grp.to_csv(OUT_DIR/"table1_by_egfr.csv", index=False)

t1_all, t1_grp


## Bước 3 — ML performance: so sánh INTRA và RING 1 3 5

In [ ]:
# Chọn bộ feature theo prefix
ROI_PREFIXES = ["intra_", "ring1_", "ring3_", "ring5_"]

def get_feature_cols(df: pd.DataFrame, prefix: str) -> list[str]:
    cols = [c for c in df.columns if c.startswith(prefix)]
    return cols

# Mô hình baseline: StandardScaler + Logistic Regression
def make_model(seed: int = SEED) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=seed))
    ])

y = cohort[LABEL_COL].astype(int).values

# Train/test split cố định để có ROC minh hoạ
train_idx, test_idx = train_test_split(
    np.arange(cohort.shape[0]),
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

results = []
roc_data = {}

for prefix in ROI_PREFIXES:
    cols = get_feature_cols(cohort, prefix)
    X = cohort[cols].values

    model = make_model(SEED)
    model.fit(X[train_idx], y[train_idx])

    prob = model.predict_proba(X[test_idx])[:, 1]
    auc = roc_auc_score(y[test_idx], prob)

    fpr, tpr, _ = roc_curve(y[test_idx], prob)
    roc_data[prefix] = (fpr, tpr, auc)

    results.append({"roi": prefix.replace("_",""), "n_features": len(cols), "auc_test": auc})

perf = pd.DataFrame(results).sort_values("auc_test", ascending=False)
perf.to_csv(OUT_DIR/"ml_performance_auc.csv", index=False)
perf


### Vẽ ROC so sánh ROI

In [ ]:
plt.figure()
for prefix, (fpr, tpr, auc) in roc_data.items():
    label = f"{prefix.replace('_','')} AUC={auc:.3f}"
    plt.plot(fpr, tpr, label=label)

plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC so sánh INTRA và RING")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR/"roc_compare_rois.png", dpi=200)
plt.show()


### Confusion matrix minh hoạ (ROI tốt nhất)

In [ ]:
best_roi = perf.iloc[0]["roi"] + "_"
best_cols = get_feature_cols(cohort, best_roi)
X_best = cohort[best_cols].values

model_best = make_model(SEED)
model_best.fit(X_best[train_idx], y[train_idx])

prob_best = model_best.predict_proba(X_best[test_idx])[:, 1]
pred_best = (prob_best >= 0.5).astype(int)

cm = confusion_matrix(y[test_idx], pred_best)
cm_df = pd.DataFrame(cm, index=["true_0","true_1"], columns=["pred_0","pred_1"])
cm_df.to_csv(OUT_DIR/"confusion_matrix_best_roi.csv", index=True)
cm_df


## Bước 4 — Stability analysis: cross validation và bootstrap CI

In [ ]:
# Cross validation AUC trên toàn bộ cohort (ROI tốt nhất)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scores = cross_val_score(make_model(SEED), X_best, y, cv=cv, scoring="roc_auc")
cv_res = pd.DataFrame({
    "fold": np.arange(1, len(scores)+1),
    "auc": scores
})
cv_summary = pd.DataFrame({
    "roi": [best_roi.replace("_","")],
    "cv_mean_auc": [scores.mean()],
    "cv_std_auc": [scores.std(ddof=1)],
    "n_splits": [cv.get_n_splits()]
})

cv_res.to_csv(OUT_DIR/"cv_folds_auc.csv", index=False)
cv_summary.to_csv(OUT_DIR/"cv_results.csv", index=False)

cv_summary


### Bootstrap CI cho AUC trên test set

In [ ]:
# Bootstrap CI trên test set (minh hoạ)
# Lưu ý: đây là CI cho AUC trên test set cố định, không phải thay thế CV.
B = 500
rng = np.random.default_rng(SEED)

y_test = y[test_idx]
p_test = prob_best

aucs = []
n = len(y_test)
for _ in range(B):
    sample_idx = rng.integers(0, n, size=n)
    if len(np.unique(y_test[sample_idx])) < 2:
        continue
    aucs.append(roc_auc_score(y_test[sample_idx], p_test[sample_idx]))

aucs = np.array(aucs)
ci_low, ci_high = np.percentile(aucs, [2.5, 97.5])

boot = pd.DataFrame({
    "roi": [best_roi.replace("_","")],
    "bootstrap_B": [B],
    "auc_mean": [aucs.mean()],
    "auc_ci_low_2p5": [ci_low],
    "auc_ci_high_97p5": [ci_high]
})
boot.to_csv(OUT_DIR/"bootstrap_ci.csv", index=False)
boot


## Bước 5 — NGS pathway: delta P và delta median P

In [ ]:
# Tạo xác suất dự đoán dạng out-of-fold (cross_val_predict) để dùng cho phân tích NGS
# Cách này giúp mỗi bệnh nhân có một dự đoán không dùng chính mình để train.

oof_prob = cross_val_predict(
    make_model(SEED),
    X_best,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

pred_df = cohort[[ID_COL, LABEL_COL]].copy()
pred_df["pred_prob"] = oof_prob

# Ghép NGS subset theo patient_id
ngs_merged = ngs64.merge(pred_df[[ID_COL, "pred_prob"]], on=ID_COL, how="inner")
print("NGS merged shape:", ngs_merged.shape)
ngs_merged.head()


### Tính delta P theo pathway

In [ ]:
# Các cột pathway dạng nhị phân (0/1) trong file demo
pathway_cols = [c for c in ngs_merged.columns if c.startswith("pathway_")]
assert len(pathway_cols) > 0

rows=[]
for pw in pathway_cols:
    mutated = ngs_merged.loc[ngs_merged[pw]==1, "pred_prob"].values
    wt = ngs_merged.loc[ngs_merged[pw]==0, "pred_prob"].values
    if len(mutated) < 5 or len(wt) < 5:
        continue

    delta_mean = mutated.mean() - wt.mean()
    delta_median = np.median(mutated) - np.median(wt)

    # Mann Whitney U (two-sided)
    pval = mannwhitneyu(mutated, wt, alternative="two-sided").pvalue

    rows.append({
        "pathway": pw.replace("pathway_",""),
        "n_mutated": len(mutated),
        "n_wt": len(wt),
        "mean_mutated": mutated.mean(),
        "mean_wt": wt.mean(),
        "delta_mean_P": delta_mean,
        "median_mutated": np.median(mutated),
        "median_wt": np.median(wt),
        "delta_median_P": delta_median,
        "p_value_mannwhitney": pval
    })

delta_table = pd.DataFrame(rows).sort_values("delta_median_P", ascending=False)
delta_table.to_csv(OUT_DIR/"ngs_deltaP_table.csv", index=False)
delta_table


### Bar chart delta median P

In [ ]:
plt.figure(figsize=(7,4))
x = delta_table["pathway"].values
y_ = delta_table["delta_median_P"].values
plt.bar(x, y_)
plt.axhline(0, linewidth=1)
plt.xticks(rotation=30, ha="right")
plt.ylabel("delta median P")
plt.title("NGS pathway: delta median P (mutated - WT)")
plt.tight_layout()
plt.savefig(OUT_DIR/"deltaP_bar.png", dpi=200)
plt.show()


## Tạo tóm tắt kết quả bằng code

In [ ]:
lines=[]
lines.append("# Kết quả tóm tắt (Day 05)")
lines.append("")
lines.append("## Cohort")
lines.append(f"- n = {cohort.shape[0]}")
lines.append(f"- EGFR dương tính = {cohort[LABEL_COL].mean()*100:.1f}%")
lines.append("")
lines.append("## ML performance (test split)")
lines.append(perf.to_markdown(index=False))
lines.append("")
lines.append("## Stability (5-fold CV)")
lines.append(cv_summary.to_markdown(index=False))
lines.append("")
lines.append("## Bootstrap CI (test split)")
lines.append(boot.to_markdown(index=False))
lines.append("")
lines.append("## NGS pathway delta P (n=64)")
lines.append(delta_table.to_markdown(index=False))
lines.append("")

summary_path = OUT_DIR/"results_summary.md"
summary_path.write_text("
".join(lines), encoding="utf-8")
print("Saved:", summary_path)


## Nén outputs để nộp

In [ ]:
import shutil
zip_path = Path("outputs/day05_submission.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path).replace(".zip",""), "zip", OUT_DIR)
print("Created:", zip_path)

# Nếu chạy trên Colab, tải file zip về máy
try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    pass
